# data cleaning practice questions

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
df =pd.read_csv("C:\\Users\\MY PC\\Downloads\\Customer_datasets - Customer_datasets.csv")
df.head(5)

,Customer_ID,Name,Age,Salary,Join_Date,Department,Performance_Score,Email,City,Gender,Is_Active
0,1357,Customer_1357,27,29664.12,2022-06-14,HR,69.1,user1356@outlook.com,Houston,Male,FALSE
1,985,Customer_985,27,47373.17,NaN,Marketing,1.9,user984@yahoo.com,Dallas,Female,FALSE
2,1828,Customer_860,62,63380.36,20170530,Operations,78.2,user859@gmail.com,NaN,Male,NaN
3,1985,Customer_1985,thirty,56746.57,2016-12-19 0:00:00,IT,85.8,user1984@company.com,New York,Male,TRUE
4,1294,Customer_1294,41,61654.21,2019-03-12,Finance,82.5,user1293@gmail.com,NEW YORK,Other,TRUE


# Basic Exploration & Missing Values

# 1. How many missing values are there in each column? Which column has the highest
# number of missing values?

In [41]:
df.isna().sum()

Customer_ID            0
Name                 145
Age                   84
Salary                97
Join_Date            254
Department            51
Performance_Score     39
Email                 49
City                  43
Gender                34
Is_Active             25
dtype: int64

In [42]:
df.isna().sum().idxmax()

'Join_Date'

# 2. What percentage of rows have at least one missing value?

In [43]:
df.isna().any(axis=1).mean() * 100

np.float64(34.87684729064039)

# 3. Drop all rows that have more than 3 missing values. How many rows remain?

In [44]:
count=df.isna().any(axis=1)
df = df[count <=3]
print(len(df))

2030


# Duplicates

# 4. How many exact duplicate rows are present in the dataset?

In [45]:
df.duplicated().sum()

np.int64(30)

# 5. How many duplicate Customer_IDs are there? Keep only the first occurrence of each Customer_ID.

In [46]:
df['Customer_ID'].unique()

array([1357,  985, 1828, ...,  861, 1460, 1127], shape=(1954,))

In [47]:
df['Customer_ID'].duplicated().sum()

np.int64(76)

In [48]:
df = df.drop_duplicates(subset='Customer_ID', keep='first')
df['Customer_ID']

0       1357
1        985
2       1828
3       1985
4       1294
        ... 
2023    1725
2024    1096
2027     861
2028    1460
2029    1127
Name: Customer_ID, Length: 1954, dtype: int64

# 6. Convert the Age column to numeric. How many values could not be converted (became NaN)?

In [50]:
df['Age']= pd.to_numeric(df['Age'],errors ='coerce')
df['Age']

0       27.0
1       27.0
2       62.0
3        NaN
4       41.0
        ... 
2023    69.0
2024    74.0
2027    21.0
2028    69.0
2029    65.0
Name: Age, Length: 1954, dtype: float64

In [21]:
df['Age'].unique()

array([ 39.,  25.,  50.,  53.,  38.,  35.,  33.,  65.,  48.,  47.,  52.,
        44.,  nan,  29.,  49.,  34.,  58.,  42.,  19.,  28.,  62.,  64.,
        21.,  26.,  57.,  22.,  59.,  54.,  46.,  31.,  67.,  24.,  68.,
        32.,  27.,  56.,  36.,  61., 100.,  60.,  43.,  20.,  66.,  51.,
        30.,  40.,  18.,  23.,  63.,  45.,  69.,  41., 150.,  37.,  55.,
       120.])

# 7. After converting Age to numeric, identify and remove outliers using the IQR method.
# How many outliers were removed?

In [51]:
Q1 = df['Age'].quantile(0.25)
Q3 = df['Age'].quantile(0.75)

IQR = Q3 - Q1
lower = Q1 - 1.5 *IQR
upper = Q3 + 1.5 *IQR
outliers = (df['Age'] < lower) | (df['Age'] > upper)

print("Outliers removed:", outliers.sum())

df = df[~outliers]

Outliers removed: 11


# 8. Replace remaining missing values in Age with the median age.

In [52]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Age'].isna().sum()

np.int64(0)

# 9. Clean the Salary column: remove currency symbols ($), commas, and convert
# everything possible to numeric. How many values remained non-numeric?

In [53]:
cleaned = (
    df['Salary'].astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)

numeric = pd.to_numeric(cleaned, errors='coerce')

print("Non-numeric values:", numeric.isna().sum())

df['Salary'] = numeric

Non-numeric values: 120


# 10. After cleaning, find the mean and median Salary. Which one is more affected by outliers?

In [55]:
df['Salary'] = df['Salary'].fillna(df['Salary'].mean())
df['Salary'] = df['Salary'].fillna(df['Salary'].median())
df['Salary'] 

0       29664.120000
1       47373.170000
2       63380.360000
3       56746.570000
4       61654.210000
            ...     
2023    41999.160000
2024    43331.220000
2027    59530.081064
2028    46179.420000
2029    58869.430000
Name: Salary, Length: 1943, dtype: float64

In [58]:
df.isna().sum().idxmax()

'Join_Date'

# 11. Convert the Join_Date column to proper datetime format. How many dates failed to parse?

In [59]:
df['Join_Date'] = pd.to_datetime(df['Join_Date'],errors ='coerce')

In [65]:
failed=df['Join_Date'].isna().sum()
print(failed)

903


# 12. Extract the year from the cleaned Join_Date. What is the most common joining year?

In [66]:
df['Join_Year'] = df['Join_Date'].dt.year

print(df['Join_Year'].mode()[0])

2019.0


# 13. Standardize the Department column (fix casing, trim spaces, fix common typos like
# “Sale” → “Sales”, “Financ” → “Finance”, etc.). How many unique departments remain after cleaning?

In [67]:
df['Department'].unique()

array(['HR', 'Marketing', 'Operations', 'IT', 'Finance', 'Sales', 'Ops',
       'Support', 'H R', nan, 'Marketing Dept', 'Unknown', 'Sale',
       'Financ', 'sales', 'SALES'], dtype=object)

In [71]:
df['Department'] = df['Department'].replace({
    'Marketing Dept':'Marketing',
    'H R':'HR',
    'Financ':'Finance',
    'Sale':'Sales',
    'Ops':'Operations',
    'sales':'Sales',
    'Unknown':'',
    
})
df['Department'].unique()

array(['HR', 'Marketing', 'Operations', 'IT', 'Finance', 'Sales',
       'Support', nan, '', 'SALES'], dtype=object)

# 14. Clean the City column (standardize names, fix “Los Angles” → “Los Angeles”, “NY” →
# “New York”, “Philly” → “Philadelphia”, etc.). List the final unique cities.

In [72]:
df['City'].unique()

array(['Houston', 'Dallas', nan, 'New York', 'NEW YORK', 'San Antonio',
       'Los Angeles', 'San Diego', 'Chicago', 'Houston TX', 'San Jose',
       'Philadelphia', 'Phoenix', 'new york', 'Philly', 'Los Angles',
       'NY', 'Unknown'], dtype=object)

In [4]:
df['City']=df['City'].replace({
    'new york':'New York',
    'NEW YORK':'New York',
    'NY':'New York',
    'Philly':'Philadelphia',
    'Los Angles':'Los Angeles',
    'Houston TX':'Houston',
    'Unknown':''
})
df['City'].unique()

array(['Houston', 'Dallas', nan, 'New York', 'San Antonio', 'Los Angeles',
       'San Diego', 'Chicago', 'San Jose', 'Philadelphia', 'Phoenix', ''],
      dtype=object)

# 15. Standardize the Gender column so it only contains “Male”, “Female”, and “Other”. How many records were changed?

In [5]:
df['Gender'].unique()

array(['Male', 'Female', 'Other', 'male', nan, 'female', 'F', 'm', 'f',
       'M', 'Unknown'], dtype=object)

In [7]:
df['Gender']= df['Gender'].replace({
    'male':'Male',
    'M':'Male',
    'm':'Male',
    'female':'Female',
    'f':'Female',
    'F':'Female',
    'nan':'Other',
    'Unknown':'Other'
})
df['Gender'].unique()

array(['Male', 'Female', 'Other', nan], dtype=object)

# 16. Convert Performance_Score to numeric. Replace non-numeric values with NaN, then
# fill missing scores with the mean.

In [8]:
df['Performance_Score']= pd.to_numeric(df['Performance_Score'], errors = 'coerce')
df['Performance_Score']

0       69.1
1        1.9
2       78.2
3       85.8
4       82.5
        ... 
2025    46.8
2026     NaN
2027    43.7
2028    17.5
2029    47.4
Name: Performance_Score, Length: 2030, dtype: float64

In [10]:
df['Performance_Score']= df['Performance_Score'].fillna(df['Performance_Score']).median()
df['Performance_Score']

0       48.2
1       48.2
2       48.2
3       48.2
4       48.2
        ... 
2025    48.2
2026    48.2
2027    48.2
2028    48.2
2029    48.2
Name: Performance_Score, Length: 2030, dtype: float64

# 17. Clean the Is_Active column so it becomes a proper boolean (True/False). Map values
# like “Yes”, “Y”, “1”, “Active”, “true” → True and “No”, “N”, “0”, “Inactive”, “false” → False.

In [11]:
df['Is_Active'].unique()

array(['FALSE', nan, 'TRUE', '0', 'Yes', '1', 'Active', 'N', 'Inactive',
       'Y', 'No'], dtype=object)

In [14]:
df['Is_Active']= df['Is_Active'].replace({
    'Yes':'TRUE',
    'Y':'TRUE',
    '1':'TRUE',
    'Active':'TRUE',
    'N':'FALSE',
    '0':'FALSE',
    'No':'FALSE',
    'Inactive':'FALSE',
    'nan':'FALSE'
})
df['Is_Active'].unique()

array(['FALSE', nan, 'TRUE'], dtype=object)

# 18. Flag invalid email addresses (those that do not contain “@” or have incomplete
# domains). How many invalid emails are there?

In [16]:
df['invalid_email'] = ~df['Email'].str.contains(
    r'^[^@\s]+@[^@\s]+\.[^@\s]+$',
    regex=True,
    na=False
)

print(df['invalid_email'].sum())

67


# 19. Clean the Name column: replace empty strings, “N/A”, and “Unknown” with NaN.
# How many names are still missing after this step?

In [17]:
df['Name'] = df['Name'].replace(['', 'N/A', 'Unknown'], np.nan)

print(df['Name'].isna().sum())

155


# 20. Perform a complete data cleaning pipeline on the entire dataset and save the
# cleaned version as cleaned_dataset.csv. Report the final shape of the cleaned dataset
# and briefly describe the main cleaning steps you applied.

In [19]:
# 1. Remove duplicate rows
df = df.drop_duplicates()

# 2. Clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# 3. Replace common missing-value indicators
df = df.replace(['', 'N/A', 'NA', 'Unknown', 'unknown', 'None'], np.nan)

# 4. Clean Name column
if 'name' in df.columns:
    df['name'] = df['name'].astype('string').str.strip()
    df['name'] = df['name'].replace(['', 'N/A', 'Unknown'], np.nan)

# 5. Clean Age
if 'age' in df.columns:
    df['age'] = (
        df['age'].astype('string')
        .str.extract(r'(\d+)', expand=False)
    )
    df['age'] = pd.to_numeric(df['age'], errors='coerce')

    # Remove unrealistic ages
    df.loc[(df['age'] < 0) | (df['age'] > 100), 'age'] = np.nan

# 6. Clean Unit Price
if 'unit_price' in df.columns:
    df['unit_price'] = (
        df['unit_price'].astype('string')
        .str.replace(r'[₹$,]', '', regex=True)
        .str.replace(r'\s*INR\s*', '', regex=True, case=False)
        .str.replace(r'(\d+(?:\.\d+)?)\s*[kK]', r'\1', regex=True)
    )

    df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')

# 7. Clean Units Sold
if 'units_sold' in df.columns:
    df['units_sold'] = (
        df['units_sold'].astype('string')
        .str.extract(r'(\d+)', expand=False)
    )
    df['units_sold'] = pd.to_numeric(df['units_sold'], errors='coerce')

# 8. Clean Discount
if 'discount_pct' in df.columns:
    df['discount_pct'] = (
        df['discount_pct'].astype('string')
        .str.replace('%', '', regex=False)
        .str.replace('percent', '', regex=False, case=False)
        .str.strip()
        .replace(['none', 'None', 'N/A'], np.nan)
    )
    df['discount_pct'] = pd.to_numeric(
        df['discount_pct'], errors='coerce'
    )

    # Keep discount between 0 and 100
    df.loc[
        (df['discount_pct'] < 0) | (df['discount_pct'] > 100),
        'discount_pct'
    ] = np.nan

# 9. Convert Order Date
if 'order_date' in df.columns:
    df['order_date'] = pd.to_datetime(
        df['order_date'], errors='coerce'
    )

# 10. Convert Join Date
if 'join_date' in df.columns:
    df['join_date'] = pd.to_datetime(
        df['join_date'], errors='coerce'
    )

# 11. Clean email addresses
if 'email' in df.columns:
    email_pattern = r'^[^@\s]+@[^@\s]+\.[^@\s]+$'

    df['invalid_email'] = ~df['email'].astype('string').str.match(
        email_pattern,
        na=False
    )

# 12. Remove completely empty rows
df = df.dropna(how='all')

# 13. Save cleaned dataset
df.to_csv('cleaned_dataset.csv', index=False)

print("Final shape:", df.shape)

Final shape: (2000, 12)


In [21]:
import os

print(os.path.exists('cleaned_dataset.csv'))

True
